# 🧪 W5-D1 概念实验：为什么"把过程写出来"能提高正确率？

> 配套阅读：`ima/第5周-Day1-思维链CoT详解.md`（CoT 定义、Zero-shot / Few-shot 对比、变体家族在那边，本 notebook 不重复）
>
> 这个 notebook 用 4 个可执行实验回答三个问题：
> 1. 跳步作答时，错误如何随步数**复利放大**？
> 2. 直接报答案 vs 分步作答，**搜索空间**差多少个数量级？
> 3. Self-consistency 多数投票，为什么"多条歪链能投出对答案"？

实验环境：纯 numpy 模拟，不调用任何 LLM。

## 实验 1：单步 90% 正确，5 步之后还剩多少？

假设模型每一步计算的隐式正确率是 90%（很不错的水平）。
- **直接作答**：5 步全部在"隐状态"里一口气完成，只有每步都隐式正确，答案才对 → $0.9^5$
- **CoT 作答**：每步写出来，写错时还有机会被"逐步自查"当场发现并改正

我们把两种模式各模拟 20 万次，看最终正确率。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def run_direct(p_step, n_steps, n_trials=200_000):
    """直接作答：n 步全部在隐状态里一次完成，只有每步都隐式正确，答案才对"""
    ok = rng.random((n_trials, n_steps)) < p_step
    return ok.all(axis=1).mean()

def run_cot(p_step, p_catch, n_steps, n_trials=200_000):
    """CoT：每步写出来并自查——写对概率 p_step；写错时以 p_catch 概率当场发现并改正"""
    ok = rng.random((n_trials, n_steps)) < p_step
    fixed = ~ok & (rng.random((n_trials, n_steps)) < p_catch)
    return (ok | fixed).all(axis=1).mean()

p, n, p_catch = 0.9, 5, 0.8
print(f"单步隐式正确率 {p:.0%} 的 {n} 步任务：")
print(f"  直接作答（跳步）        理论 {p**n:.1%} | 模拟 {run_direct(p, n):.1%}")
print(f"  CoT 分步 + 逐步自查({p_catch:.0%}) 理论 {(1-(1-p)*(1-p_catch))**n:.1%} | 模拟 {run_cot(p, p_catch, n):.1%}")
print()
print("结论：错误是乘法累积的。CoT 的价值之一 = 给每步一次'被发现、被改正'的机会。")

## 实验 2：搜索空间的爆炸——直接蒙 vs 逐步选

一道 5 步题，每步有 10 个候选动作。正确答案是一条**完整路径**。
- 直接报答案 ≈ 从 $10^5 = 100{,}000$ 条完整路径里一次性蒙中
- 分步作答：每步只在 10 个候选里选 1 个，还能逐步代入检验 → 总共只需评估 $5 \times 10 = 50$ 个候选

In [ ]:
import numpy as np

rng = np.random.default_rng(1)

B, D = 10, 5                      # 每步 10 个候选，共 5 步
target = rng.integers(0, B, D)    # 唯一正确的完整组合

trials = 200_000
guess = rng.integers(0, B, (trials, D))
hit = (guess == target).all(axis=1).mean()

print(f"{D} 步 × 每步 {B} 个候选 = {B**D:,} 条完整路径")
print(f"直接作答：一次蒙中完整路径的概率 ≈ {1/B**D:.5%}（模拟 {hit:.5%}）")
print(f"分步作答：每步 10 选 1、可局部验证 → 共评估 {D}×{B} = {D*B} 个候选（少 {B**D/(D*B):,.0f} 倍）")
print()
print("CoT 的第二个本质：把指数级的联合搜索，拆成 D 个可局部检验的小搜索。")

## 实验 3：Self-consistency——多条"歪链"投票

单条推理链正确率只有 p，答错时均匀散落在 9 个错误选项上。
采样 k 条链、对**最终答案**（不是过程）投票，正确率如何随 k 变化？
直觉：只要"正确阵营"平均人数 > 任何"错误阵营"，票数越多优势越稳。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

rng = np.random.default_rng(7)

def vote_accuracy(p_correct, n_wrong, k, n_problems=20_000):
    """每条链以 p_correct 概率答对；否则均匀落在 n_wrong 个错误选项上。多数投票"""
    chains = np.where(rng.random((n_problems, k)) < p_correct, 0,
                      1 + rng.integers(0, n_wrong, (n_problems, k)))
    counts = np.zeros((n_problems, n_wrong + 1))
    for j in range(n_wrong + 1):
        counts[:, j] = (chains == j).sum(axis=1)
    return (counts.argmax(axis=1) == 0).mean()

ks = list(range(1, 22, 2))
plt.figure(figsize=(7, 4))
for p in (0.5, 0.6, 0.7):
    accs = [vote_accuracy(p, 9, k) for k in ks]
    plt.plot(ks, [a * 100 for a in accs], "o-", label=f"单链正确率 {p:.0%}")
plt.xlabel("采样链数 k")
plt.ylabel("多数投票后的正确率 (%)")
plt.title("Self-consistency：错误分散在 9 个选项，正确答案天然是多数派")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("单链 60% 的模型，11 条链投票后能到 ~95% —— 前提：错误答案彼此不同（分散）。")
print("如果所有错误链都错向同一个选项（系统性偏差），投票也救不回来。")

## 实验 4：CoT 的增益随任务步数扩大

把实验 1 的两条公式画成曲线：步数越多，"跳步"掉得越快，CoT 的相对优势越大。
这解释了为什么 CoT 在 GSM8K 这类多步题上提升最大，而一步事实题几乎无感。

In [ ]:
import numpy as np
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

p_step, p_catch = 0.9, 0.8
steps = np.arange(1, 11)
direct = p_step ** steps
cot = (1 - (1 - p_step) * (1 - p_catch)) ** steps

plt.figure(figsize=(7, 4))
plt.plot(steps, direct * 100, "s-", label="直接作答（跳步）")
plt.plot(steps, cot * 100, "o-", label="CoT（分步 + 逐步自查）")
plt.xlabel("任务的推理步数")
plt.ylabel("最终答案正确率 (%)")
plt.title("步数越多，CoT 的相对优势越大")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"10 步任务：直接作答 {direct[9]:.1%} vs CoT {cot[9]:.1%}")

## 结论

| 实验 | 看到了什么 |
|---|---|
| 1 | 单步 90%，5 步只剩 59%；逐步自查把它拉回 90% |
| 2 | 直接报答案 = 从 100,000 条路径蒙 1 条；分步 = 50 个可检验的小选择 |
| 3 | 单链 60% → 11 链投票 ~95%（错误分散是前提） |
| 4 | 步数越多，CoT 相对优势越大 → 多步任务才值得开 CoT |

→ 深入阅读：`ima/第5周-Day1-思维链CoT详解.md`（Zero-shot/Few-shot CoT、Auto-CoT 等变体、常见误区）